# IMPPORT ESSENTIAL LIBRARIES
uninstall conflicting packages and reinstall the exact versions you used before.

In [2]:
# 1️⃣ Remove conflicting libraries
!pip uninstall -y tensorflow tensorflow-estimator keras jax jaxlib numpy

# 2️⃣ Reinstall correct versions
!pip install --no-cache-dir numpy==1.24.3 tensorflow==2.13.0 tensorflow-estimator==2.13.0 keras==2.13.1

# 3️⃣ Restart runtime automatically (so TF reloads clean)
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: keras 3.8.0
Uninstalling keras-3.8.0:
  Successfully uninstalled keras-3.8.0
Found existing installation: jax 0.5.2
Uninstalling jax-0.5.2:
  Successfully uninstalled jax-0.5.2
Found existing installation: jaxlib 0.5.1
Uninstalling jaxlib-0.5.1:
  Successfully uninstalled jaxlib-0.5.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 276.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 524.2/524.2 MB 206.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.8/440.8 kB 322.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 319.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 207.9 MB/s eta 0:00:00
  Attempting uninstall: typing-exten

{'status': 'ok', 'restart': True}

In [1]:
# (After restart you should re-run the imports cell below)
!pip install --upgrade --no-cache-dir typing_extensions
!pip install --quiet noisereduce soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.5.0
    Uninstalling typing_extensions-4.5.0:
      Successfully uninstalled typing_extensions-4.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
orbax-checkpoint 0.11.16 requires jax>=0.5.0, which is not installed.
flax 0.10.6 requires jax>=0.5.1, which is not installed.
chex 0.1.89 requires jax>=0.4.27, which is not installed.
chex 0.1.89 requires jaxlib>=0.4.27, which is not installed.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tensorflow 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing

In [2]:
import numpy as np, tensorflow as tf
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)

import torch
import torchaudio
import librosa

import pickle
from tqdm import tqdm

import os, glob
import pandas as pd
import numpy as np
import random

from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import Sequence
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import normalize

from tensorflow.keras.optimizers import Adam

import noisereduce as nr
import soundfile as sf

# set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

NumPy: 1.24.3
TensorFlow: 2.13.0


# DATA OPTIMIZATION

## inforamtion about data

In [3]:
DATA_DIR = "/kaggle/input/speaker-recognition-cmu-arctic/train"
rows = []

for sp in sorted(os.listdir(DATA_DIR)):
    spath = os.path.join(DATA_DIR, sp)
    if not os.path.isdir(spath): continue
    wavs = glob.glob(os.path.join(spath, "*.wav"))
    for w in wavs:
        try:
            y, sr = librosa.load(w, sr=None)
            dur = librosa.get_duration(y=y, sr=sr)
        except:
            dur = None
        rows.append((sp, w, dur))
        
df = pd.DataFrame(rows, columns=["speaker","path","duration"])
print("Speakers:", df["speaker"].nunique())
print("Files per speaker:\n", df["speaker"].value_counts())

Speakers: 18
Files per speaker:
 speaker
awb    910
jmk    909
aew    906
clb    906
rms    906
ksp    906
slt    906
bdl    906
lnh    905
rxr    533
eey    475
fem    474
ahw    474
axb    474
ljm    474
aup    474
slp    474
gka    474
Name: count, dtype: int64


In [4]:
print("Durations stats:\n", df["duration"].describe())

Durations stats:
 count    12486.000000
mean         3.206057
std          0.918252
min          0.925000
25%          2.545062
50%          3.155063
75%          3.785000
max          7.665063
Name: duration, dtype: float64


## make all records be 3 sec

In [5]:
INPUT_DIR = "/kaggle/input/speaker-recognition-cmu-arctic/train"
OUTPUT_DIR = "preprocessed_train"

# same sample rate and duration
SAMPLE_RATE = 16000
TARGET_LEN = SAMPLE_RATE * 3

processed = 0
skipped = 0

for speaker in os.listdir(INPUT_DIR):
    speaker_path = os.path.join(INPUT_DIR, speaker)
    if not os.path.isdir(speaker_path):
        continue

    # folder for each speaker
    speaker_out = os.path.join(OUTPUT_DIR, speaker)
    os.makedirs(speaker_out, exist_ok=True)

    for f in os.listdir(speaker_path):
        file_path = os.path.join(speaker_path, f)

        try:
            signal, sr = torchaudio.load(file_path)

            # resample if needed
            if sr != SAMPLE_RATE:
                signal = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)(signal)

            duration = signal.size(1) / SAMPLE_RATE

            if duration < 1.0:
                print(f" Skipped (too short): {f} ({duration:.2f}s)")
                skipped += 1
                continue

            # padding
            if signal.size(1) < TARGET_LEN:
                pad_len = TARGET_LEN - signal.size(1)
                signal = torch.nn.functional.pad(signal, (0, pad_len))
                print(f" Padded: {f} ({duration:.2f}s → 3s)")

            # crop if more than 3s
            else:
                signal = signal[:, :TARGET_LEN]
                print(f" Cropped: {f} ({duration:.2f}s → 3s)")

            # save after preprocessing (this is the simple crop/pad stage)
            out_path = os.path.join(speaker_out, f)
            torchaudio.save(out_path, signal, SAMPLE_RATE)
            processed += 1

        except Exception as e:
            print(f" Error in {file_path}: {e}")

print(f"\n Processing finished! {processed} files saved in '{OUTPUT_DIR}', Skipped: {skipped}")

 Cropped: arctic_a0095.wav (4.26s → 3s)
 Padded: arctic_a0306.wav (2.98s → 3s)
 Cropped: arctic_a0051.wav (4.12s → 3s)
 Cropped: arctic_a0518.wav (3.00s → 3s)
 Padded: arctic_a0322.wav (2.81s → 3s)
 Padded: arctic_a0330.wav (2.92s → 3s)
 Cropped: arctic_a0349.wav (3.72s → 3s)
 Padded: arctic_a0173.wav (2.71s → 3s)
 Cropped: arctic_a0501.wav (4.01s → 3s)
 Padded: arctic_a0418.wav (2.74s → 3s)
 Padded: arctic_a0301.wav (2.72s → 3s)
 Cropped: arctic_a0233.wav (3.29s → 3s)
 Cropped: arctic_a0277.wav (4.51s → 3s)
 Cropped: arctic_a0168.wav (3.67s → 3s)
 Cropped: arctic_a0424.wav (4.20s → 3s)
 Cropped: arctic_a0472.wav (5.03s → 3s)
 Padded: arctic_a0122.wav (2.85s → 3s)
 Padded: arctic_a0409.wav (2.56s → 3s)
 Cropped: arctic_a0188.wav (3.27s → 3s)
 Cropped: arctic_a0006.wav (3.25s → 3s)
 Cropped: arctic_a0250.wav (5.53s → 3s)
 Padded: arctic_a0356.wav (2.78s → 3s)
 Cropped: arctic_a0451.wav (3.72s → 3s)
 Cropped: arctic_a0500.wav (3.90s → 3s)
 Cropped: arctic_a0575.wav (3.70s → 3s)
 Padded: 

In [6]:
DATA_DIR = "/kaggle/input/speaker-recognition-cmu-arctic/test"
OUTPUT_DIR = "preprocessed_test"

SAMPLE_RATE = 16000
TARGET_LEN = SAMPLE_RATE * 3
os.makedirs(OUTPUT_DIR, exist_ok=True)

processed = 0
skipped = 0

for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if not f.endswith(".wav"):
            continue

        path = os.path.join(root, f)
        signal, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
            signal = resampler(signal)

        duration = signal.size(1) / SAMPLE_RATE

        if duration < 1.0:
            print(f" Skipped (too short): {f} ({duration:.2f}s)")
            skipped += 1
            continue

        if signal.size(1) < TARGET_LEN:
            # padding
            pad_len = TARGET_LEN - signal.size(1)
            signal = torch.nn.functional.pad(signal, (0, pad_len))
            
        else:
            # crop
            signal = signal[:, :TARGET_LEN]

        out_path = os.path.join(OUTPUT_DIR, f)
        torchaudio.save(out_path, signal, SAMPLE_RATE)
        processed += 1

 Cropped: CDBK21190472333548.wav (3.60s → 3s)
 Padded: XKQP35918600326135.wav (2.21s → 3s)
 Cropped: VMDF13298681926195.wav (3.25s → 3s)
 Padded: HBNG93880431156761.wav (2.12s → 3s)
 Padded: MRPQ70083135550803.wav (2.92s → 3s)
 Cropped: FQIQ55388965219586.wav (3.45s → 3s)
 Cropped: HAXS98131851215726.wav (3.73s → 3s)
 Padded: SNGI37561323315110.wav (2.22s → 3s)
 Cropped: RMPZ52385837079958.wav (5.11s → 3s)
 Cropped: OEUO90021824730237.wav (3.67s → 3s)
 Padded: BHOI53545903254710.wav (2.71s → 3s)
 Cropped: IPVA96902081081101.wav (4.33s → 3s)
 Cropped: KEWB81814152183365.wav (3.73s → 3s)
 Cropped: CJAZ76020257148331.wav (3.29s → 3s)
 Cropped: TEXI73706670090206.wav (3.05s → 3s)
 Padded: ODVG19388574978042.wav (1.82s → 3s)
 Cropped: OVGB49973977166608.wav (3.91s → 3s)
 Cropped: APQC41609456884992.wav (3.05s → 3s)
 Padded: ANKS73776413254295.wav (2.26s → 3s)
 Cropped: VOVB04344857970405.wav (3.12s → 3s)
 Padded: KJXM56485699127338.wav (1.81s → 3s)
 Cropped: TUOD23349254803230.wav (3.08s → 

In [7]:
import IPython.display as ipd

# try random file to ensure from preprocessing (path matches where preprocessed saved)
sample_check_path = "/kaggle/working/preprocessed_train"
# Example (uncomment if exists in your env):
audio, sr = librosa.load("/kaggle/working/preprocessed_train/aew/arctic_a0003.wav", sr=16000)
print(audio.shape, sr)
ipd.Audio(audio, rate=sr)

(48000,) 16000


In [8]:
test_meta = pd.read_csv("/kaggle/input/speaker-recognition-cmu-arctic/test_full.csv")
train_meta  = pd.read_csv("/kaggle/input/speaker-recognition-cmu-arctic/train.csv")

print("Train.csv:")
print(train_meta.head())

print("\nTest_full.csv:")
print(test_meta.head())

print("\nTrain columns:", train_meta.columns.tolist())
print("Test columns:", test_meta.columns.tolist())

Train.csv:
          id                   file_path  \
0  rxr_a0591  train/rxr/arctic_a0591.wav   
1  rxr_a0403  train/rxr/arctic_a0403.wav   
2  ljm_a0059  train/ljm/arctic_a0059.wav   
3  jmk_a0134  train/jmk/arctic_a0134.wav   
4  rms_b0067  train/rms/arctic_b0067.wav   

                                              speech speaker  
0                     We are both children together.     rxr  
1    His newborn cunning gave him poise and control.     rxr  
2                His immaculate appearance was gone.     ljm  
3                He obeyed the pressure of her hand.     jmk  
4  Below him the shadow was broken into a pool of...     rms  

Test_full.csv:
                   id                    file_path  \
0  OELV49874079341496  test/OELV49874079341496.wav   
1  ELDT67178766030957  test/ELDT67178766030957.wav   
2  GWMS82652863581753  test/GWMS82652863581753.wav   
3  UAPD90278002373588  test/UAPD90278002373588.wav   
4  IHIY50549582963537  test/IHIY50549582963537.wav   

     

## AUDIO ENHANCEMENT (DENOISING & AUGMENTATION)
After making all records 3s, we perform:
- silence trimming (librosa.effects.split) on the preprocessed files (extra trimming)
- noise reduction using `noisereduce`
- precompute augmentations (noise, pitch, time-stretch, time-shift)
Augmented files will be stored under `augmented_train/<speaker>/`.

In [9]:
TOP_DB = 20
ENERGY_THRESHOLD = 1e-4

# Enhancement & augmentation parameters
ENHANCED_DIR = "enhanced_train"       # cleaned (denoised + trimmed) files
AUGMENTED_DIR = "augmented_train"     # augmented copies saved here
os.makedirs(ENHANCED_DIR, exist_ok=True)
os.makedirs(AUGMENTED_DIR, exist_ok=True)

# helper conversions
def tensor_to_numpy_mono(tensor):
    arr = tensor.numpy()
    if arr.ndim == 2:
        arr = arr.mean(axis=0)   # convert to mono by averaging channels
    return arr

def numpy_to_tensor(y):
    if isinstance(y, np.ndarray):
        t = torch.from_numpy(y.astype(np.float32))
        if t.ndim == 1:
            t = t.unsqueeze(0)  # (1, samples)
        return t
    else:
        return y

# preprocessing helpers (silence removal, denoise, energy check)
def remove_silence_numpy(y, sr, top_db=20):
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return None
    y_trimmed = np.concatenate([y[start:end] for start, end in intervals])
    return y_trimmed

def denoise_numpy(y, sr):
    try:
        y_out = nr.reduce_noise(y=y, sr=sr)
        return y_out
    except Exception:
        return y

def has_voice_numpy(y, energy_threshold=1e-4):
    energy = np.mean(y**2)
    return energy > energy_threshold

def pad_or_crop_numpy(y, target_len=TARGET_LEN):
    if len(y) < target_len:
        return np.pad(y, (0, target_len - len(y)))
    else:
        return y[:target_len]

# Augmentation helpers (precompute versions)
def add_noise_numpy(y, snr_db):
    rms = np.sqrt(np.mean(y ** 2))
    snr = 10 ** (snr_db / 20.0)
    noise_rms = rms / snr
    noise = np.random.normal(0, noise_rms, size=y.shape)
    return y + noise

def time_stretch_numpy(y, rate):
    try:
        y_s = librosa.effects.time_stretch(y, rate)
    except Exception:
        return y
    if len(y_s) < TARGET_LEN:
        y_s = np.pad(y_s, (0, TARGET_LEN - len(y_s)))
    else:
        y_s = y_s[:TARGET_LEN]
    return y_s

def time_shift_numpy(y, shift_max=0.1):
    """
    Shift audio in time by up to shift_max fraction of length (positive or negative).
    """
    shift = int(random.uniform(-shift_max, shift_max) * len(y))
    if shift == 0:
        return y
    if shift > 0:
        y_s = np.pad(y, (shift, 0))[:len(y)]
    else:
        y_s = np.pad(y, (0, -shift))[ -shift: ] if -shift < len(y) else np.zeros_like(y)
    return y_s

In [10]:
# iterate over preprocessed files created earlier in preprocessed_train and produce enhanced + augmented copies
aug_count = 0
enhanced_count = 0

for speaker in sorted(os.listdir("preprocessed_train")):
    in_spath = os.path.join("preprocessed_train", speaker)
    if not os.path.isdir(in_spath):
        continue
    out_spath = os.path.join(ENHANCED_DIR, speaker)
    os.makedirs(out_spath, exist_ok=True)
    aug_spath = os.path.join(AUGMENTED_DIR, speaker)
    os.makedirs(aug_spath, exist_ok=True)

    for f in sorted(os.listdir(in_spath)):
        src = os.path.join(in_spath, f)
        try:
            # load (preprocessed files are already 16k mono shape (1, samples) or (samples,))
            y, sr = librosa.load(src, sr=SAMPLE_RATE)
            
            # 1) extra silence trimming
            y_trim = remove_silence_numpy(y, sr, top_db=TOP_DB)
            if y_trim is None:
                # if silence removal removes everything, skip
                continue
            
            # 2) denoise
            y_dn = denoise_numpy(y_trim, sr)
            
            # 3) energy check
            if not has_voice_numpy(y_dn, ENERGY_THRESHOLD):
                continue
            
            # 4) pad/crop back to fixed length
            y_fixed = pad_or_crop_numpy(y_dn, TARGET_LEN)
            
            # save enhanced (cleaned) file
            enhanced_name = f  # same filename kept
            enhanced_path = os.path.join(out_spath, enhanced_name)
            sf.write(enhanced_path, y_fixed, SAMPLE_RATE)
            enhanced_count += 1

            # 5) create augmentations (precompute)
            # noise
            y_noise = add_noise_numpy(y_fixed, snr_db=random.uniform(5, 20))
            noise_name = f.replace(".wav", "_noise.wav")
            sf.write(os.path.join(aug_spath, noise_name), y_noise, SAMPLE_RATE)
            
            # time-stretch
            y_stretch = time_stretch_numpy(y_fixed, rate=random.uniform(0.9, 1.1))
            stretch_name = f.replace(".wav", "_stretch.wav")
            sf.write(os.path.join(aug_spath, stretch_name), y_stretch, SAMPLE_RATE)
            
            # time-shift
            y_shift = time_shift_numpy(y_fixed, shift_max=0.05)
            shift_name = f.replace(".wav", "_shift.wav")
            sf.write(os.path.join(aug_spath, shift_name), y_shift, SAMPLE_RATE)

            aug_count += 4

            if aug_count % 500 == 0:
                print(f"Augmented files created: {aug_count}")

        except Exception as e:
            print(f"Aug error for {src}: {e}")

print(f"Enhancement finished. Enhanced: {enhanced_count}, Augmented files created: {aug_count}")

Augmented files created: 500
Augmented files created: 1000
Augmented files created: 1500
Augmented files created: 2000
Augmented files created: 2500
Augmented files created: 3000
Augmented files created: 3500
Augmented files created: 4000
Augmented files created: 4500
Augmented files created: 5000
Augmented files created: 5500
Augmented files created: 6000
Augmented files created: 6500
Augmented files created: 7000
Augmented files created: 7500
Augmented files created: 8000
Augmented files created: 8500
Augmented files created: 9000
Augmented files created: 9500
Augmented files created: 10000
Augmented files created: 10500
Augmented files created: 11000
Augmented files created: 11500
Augmented files created: 12000
Augmented files created: 12500
Augmented files created: 13000
Augmented files created: 13500
Augmented files created: 14000
Augmented files created: 14500
Augmented files created: 15000
Augmented files created: 15500
Augmented files created: 16000
Augmented files created: 165

In [15]:
import IPython.display as ipd

# try random file to ensure from preprocessing (path matches where preprocessed saved)
sample_check_path = "/kaggle/working/preprocessed_train"
# Example (uncomment if exists in your env):
audio, sr = librosa.load("/kaggle/working/augmented_train/aew/arctic_a0003_stretch.wav", sr=16000)
print(audio.shape, sr)
ipd.Audio(audio, rate=sr)

(48000,) 16000


## Build combined CSV of (file_path, speaker) for preprocessed (enhanced) + augmented
We'll create a CSV similar to your original `train.csv` format but pointing to the cleaned/augmented files (absolute paths).
This will be used by `extract_features_from_csv` to compute MFCCs.

In [12]:
combined_rows = []

# enhanced (cleaned) files
for speaker in sorted(os.listdir(ENHANCED_DIR)):
    spath = os.path.join(ENHANCED_DIR, speaker)
    if not os.path.isdir(spath):
        continue
    for f in sorted(os.listdir(spath)):
        combined_rows.append((speaker, os.path.join(spath, f)))

# augmented
for speaker in sorted(os.listdir(AUGMENTED_DIR)):
    spath = os.path.join(AUGMENTED_DIR, speaker)
    if not os.path.isdir(spath):
        continue
    for f in sorted(os.listdir(spath)):
        combined_rows.append((speaker, os.path.join(spath, f)))

combined_df = pd.DataFrame(combined_rows, columns=["speaker", "file_path"])
combined_csv_path = "combined_preprocessed_augmented.csv"
combined_df.to_csv(combined_csv_path, index=False)

print("Combined CSV saved to:", combined_csv_path)
print("Total files in combined:", len(combined_df))

Combined CSV saved to: combined_preprocessed_augmented.csv
Total files in combined: 49932


# FEATURE EXTRACTION (MFCC)
We reuse your original style `extract_features_from_csv` but adapt to accept absolute paths in file_path column.
We will extract MFCC (n_mfcc=40) and produce mean-pooled vectors for each file.

In [13]:
def extract_features_from_csv(csv_file, base_dir=None, sr=16000, duration=3, n_mfcc=40):
    
    df = pd.read_csv(csv_file)
    features = []
    labels = []
    
    for _, row in df.iterrows():
        # file_path is absolute (we saved absolute paths in combined CSV)
        file_path = row["file_path"] if "file_path" in row else os.path.join(base_dir, row["file_path"])
        speaker   = row["speaker"]

        try:
            y, _ = librosa.load(file_path, sr=sr, duration=duration)
            
            if len(y) < sr * duration:
                pad_len = sr * duration - len(y)
                y = np.pad(y, (0, pad_len), mode="constant")
            
            # MFCC extraction (same as original)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
            mfcc = np.mean(mfcc.T, axis=0)

            features.append(mfcc)
            labels.append(speaker)
        
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return np.array(features), np.array(labels)

In [16]:
# Extract features from combined CSV (enhanced + augmented)
X_train, y_train = extract_features_from_csv(combined_csv_path, sr=SAMPLE_RATE, duration=3, n_mfcc=40)

# Optionally extract test features from preprocessed_test if desired (left as original behavior)

# SPLIT DATA AND PERPROCESSING

In [17]:
encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train)
# if you had test labels you would transform similarly:
# y_test_encoded = encoder.transform(y_test)

In [18]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train_encoded.shape)

# reshape to match your original model input expectation (features, 1)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

X_train shape: (49932, 40)
y_train shape: (49932,)


# MODEL SET UP

## primary layers Conv1D

In [19]:
from tensorflow.keras import layers, models, regularizers

def create_embedding_model_with_cnn(input_shape, embedding_dim=256):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.25),

        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),

        layers.Conv1D(256, kernel_size=3, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(), 

        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(embedding_dim),
        layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))
    ])
    return model

## set triplet loss

In [20]:
import tensorflow.keras.backend as K

def triplet_loss(alpha=0.2):
    def loss(y_true, y_pred):
        total_leng = y_pred.shape.as_list()[-1] // 3

        anchor   = y_pred[:, 0:total_leng]
        positive = y_pred[:, total_leng:2*total_leng]
        negative = y_pred[:, 2*total_leng:]

        pos_dist = K.sum(K.square(anchor - positive), axis=1)
        neg_dist = K.sum(K.square(anchor - negative), axis=1)

        basic_loss = pos_dist - neg_dist + alpha
        loss = K.maximum(basic_loss, 0.0)
        return loss
    return loss

## embedding

In [21]:
import tensorflow as tf

def triplet_accuracy_metric(y_true, y_pred):
  
    # (batch_size, embedding_dim*3)
    embedding_dim = tf.shape(y_pred)[1] // 3
    anchor = y_pred[:, :embedding_dim]
    positive = y_pred[:, embedding_dim:2*embedding_dim]
    negative = y_pred[:, 2*embedding_dim:]

    # calculate distances
    d_pos = tf.norm(anchor - positive, axis=1)
    d_neg = tf.norm(anchor - negative, axis=1)

    # check triplet
    correct = tf.cast(d_pos < d_neg, tf.float32)
    return tf.reduce_mean(correct)

In [22]:
def build_siamese_model(input_shape, embedding_dim=256, alpha=0.2):
    embedding_model = create_embedding_model_with_cnn(input_shape, embedding_dim)

    input_anchor   = layers.Input(shape=input_shape, name="anchor")
    input_positive = layers.Input(shape=input_shape, name="positive")
    input_negative = layers.Input(shape=input_shape, name="negative")

    emb_anchor   = embedding_model(input_anchor)
    emb_positive = embedding_model(input_positive)
    emb_negative = embedding_model(input_negative)

    merged_output = layers.concatenate([emb_anchor, emb_positive, emb_negative], axis=1)

    model = models.Model(inputs=[input_anchor, input_positive, input_negative], outputs=merged_output)
    
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss=triplet_loss(alpha),
        metrics=[triplet_accuracy_metric]
    )
    return model, embedding_model

## **Summary**

build_siamese_model constructs a model that takes three audio recordings as input: an Anchor (A), a Positive (P), and a Negative (N).

Each input recording is passed through the same shared network to produce an embedding (a fixed-size vector representation).

We combine the three embeddings and compute the Triplet Loss on them during training.

After training, we extract and use the embedding_model (the shared network alone) as a tool to convert any audio recording into its embedding vector.

To perform speaker verification or identification, we compare embeddings (e.g., using cosine similarity or Euclidean distance) to decide whether two recordings belong to the same speaker.

In [23]:
import random

def generate_triplets(features, labels, num_triplets=10000):
    
    triplets = []
    unique_labels = np.unique(labels)

    for _ in range(num_triplets):
        pos_label = random.choice(unique_labels)

        pos_indices = np.where(labels == pos_label)[0]
        anchor_idx, positive_idx = np.random.choice(pos_indices, 2, replace=True)

        neg_label = random.choice(unique_labels[unique_labels != pos_label])
        neg_indices = np.where(labels == neg_label)[0]
        negative_idx = np.random.choice(neg_indices)

        triplets.append([features[anchor_idx], features[positive_idx], features[negative_idx]])

    A, P, N = zip(*triplets)
    return np.array(A), np.array(P), np.array(N)

In [24]:
# Generate triplets from X_train (which includes enhanced + augmented data)
A_train, P_train, N_train = generate_triplets(X_train, y_train_encoded, num_triplets=30000)

## Train/Evaluate/triplet loss accuracy

In [25]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

earlyStop = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True)

reduceLR = ReduceLROnPlateau(
    monitor='val_loss', patience=7, factor=0.1)

modelPoints = ModelCheckpoint(
    '/kaggle/working/model.h5', save_best_only=True)

In [ ]:
input_shape = A_train.shape[1:]
siamese_model, embedding_model = build_siamese_model(input_shape)

history = siamese_model.fit(
    [A_train, P_train, N_train],
    np.zeros(len(A_train)),
    batch_size=64,
    validation_split=0.2,
    epochs=110,
    callbacks=[earlyStop,reduceLR,modelPoints]
)

Epoch 1/110
375/375 [==============================] - 17s 33ms/step - loss: 0.1565 - triplet_accuracy_metric: 0.7335 - val_loss: 0.1075 - val_triplet_accuracy_metric: 0.8424 - lr: 1.0000e-04
Epoch 2/110
  5/375 [..............................] - ETA: 11s - loss: 0.1366 - triplet_accuracy_metric: 0.7625

/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


375/375 [==============================] - 12s 31ms/step - loss: 0.1252 - triplet_accuracy_metric: 0.7978 - val_loss: 0.0966 - val_triplet_accuracy_metric: 0.8584 - lr: 1.0000e-04
Epoch 3/110
375/375 [==============================] - 12s 31ms/step - loss: 0.1138 - triplet_accuracy_metric: 0.8212 - val_loss: 0.0901 - val_triplet_accuracy_metric: 0.8739 - lr: 1.0000e-04
Epoch 4/110
375/375 [==============================] - 12s 32ms/step - loss: 0.1035 - triplet_accuracy_metric: 0.8395 - val_loss: 0.0841 - val_triplet_accuracy_metric: 0.8896 - lr: 1.0000e-04
Epoch 5/110
375/375 [==============================] - 12s 32ms/step - loss: 0.0961 - triplet_accuracy_metric: 0.8556 - val_loss: 0.0781 - val_triplet_accuracy_metric: 0.8976 - lr: 1.0000e-04
Epoch 6/110
375/375 [==============================] - 12s 32ms/step - loss: 0.0890 - triplet_accuracy_metric: 0.8704 - val_loss: 0.0728 - val_triplet_accuracy_metric: 0.9094 - lr: 1.0000e-04
Epoch 7/110
375/375 [==============================]

## save model

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger
logger = CSVLogger('/kaggle/working/loggerfile.csv')

# Save embedding model after training (call .save after fit if you want)
embedding_model.save("embedding_model.h5")
print("Saved embedding_model.h5")

## Final notes
- This notebook preserves your original model structure and variable naming as much as possible.
- Preprocessing (silence removal + denoise) is applied in a separate enhancement step and augmented files are precomputed.
- The `combined_preprocessed_augmented.csv` file contains absolute paths and is used by `extract_features_from_csv` to produce MFCC vectors used for training.
- If anything fails on Kaggle regarding paths or package installs, paste the error and I will adjust the notebook accordingly.